# Lecture 9 Demo 3: When Polars Is Worth It

Aligned with the **When Polars Is Worth It** section of Lecture 9.

## What this notebook covers
- when pandas is still the practical default
- when a lazy Polars scan is worth trying
- how window expressions solve per-group ranking problems
- how to hand a Polars result back to pandas when needed

The point is not to replace pandas everywhere. The point is to recognize workloads where Polars earns the extra mental model.

In [5]:
from __future__ import annotations

import pandas as pd
import polars as pl

from demo_utils import data_path, section

section("Versions")
print(f"pandas: {pd.__version__}")
print(f"polars: {pl.__version__}")
print("This notebook focuses on when Polars is worth the extra mental model.")


Versions
pandas: 3.0.2
polars: 1.39.3
This notebook focuses on when Polars is worth the extra mental model.


## 1. Large CSV, small answer table

This is the classic Polars pitch:
- read a wide CSV lazily with `scan_csv(...)`
- keep only needed columns
- filter before grouping
- materialize the answer only at the end

On a small classroom dataset pandas would also work. The point is to see the pattern that becomes valuable as files grow.

In [6]:
section("Large CSV, small answer table")

quake_query = (
    pl.scan_csv(data_path("earthquakes.csv"))
    .select("net", "mag", "tsunami")
    .filter(pl.col("mag").cast(pl.Float64) >= 6.0)
    .group_by("net")
    .agg(
        pl.len().alias("quake_count"),
        pl.col("mag").cast(pl.Float64).mean().alias("avg_magnitude"),
        pl.col("tsunami").cast(pl.Int64).sum().alias("tsunami_events"),
    )
    .sort("quake_count", descending=True)
)

print("Optimized plan:")
print(quake_query.explain())

quake_summary = quake_query.collect()
print("\nMaterialized summary:")
print(quake_summary.head(10))


Large CSV, small answer table
Optimized plan:
SORT BY [descending: [true]] [col("quake_count")]
  AGGREGATE[maintain_order: false]
    [len().alias("quake_count"), col("mag").mean().alias("avg_magnitude"), col("tsunami").sum().alias("tsunami_events")] BY [col("net")]
    FROM
    simple π 3/3 ["net", "mag", "tsunami"]
      Csv SCAN [/Users/zoltankegli/work/elte/python-course-2026/9_AI_ML_DS_1/src/earthquakes.csv]
      PROJECT 3/28 COLUMNS
      SELECTION: [(col("mag")) >= (6.0)]
      ESTIMATED ROWS: 1234

Materialized summary:
shape: (7, 4)
┌─────┬─────────────┬───────────────┬────────────────┐
│ net ┆ quake_count ┆ avg_magnitude ┆ tsunami_events │
│ --- ┆ ---         ┆ ---           ┆ ---            │
│ str ┆ u32         ┆ f64           ┆ i64            │
╞═════╪═════════════╪═══════════════╪════════════════╡
│ us  ┆ 1324        ┆ 6.377719      ┆ 415            │
│ ak  ┆ 8           ┆ 6.75          ┆ 8              │
│ nc  ┆ 4           ┆ 6.4           ┆ 3              │
│ pr  ┆ 2

## 2. Rank within each agency, then hand off to pandas

The next cell shows two practical ideas:

1. rank rows inside each agency with a window expression
2. move the Polars result to pandas when plotting, scikit-learn, or older code expects it

That second point matters in real projects: tool choice is contextual, not ideological.

In [7]:
section("Rank within each agency")

missions = pl.read_csv(data_path("space_missions.csv"))

ranked_missions = (
    missions.select("agency", "mission_name", "cost_million_usd")
    .with_columns(
        pl.col("cost_million_usd")
        .rank("dense", descending=True)
        .over("agency")
        .alias("cost_rank_within_agency")
    )
    .filter(pl.col("cost_rank_within_agency") <= 2)
    .sort(["agency", "cost_rank_within_agency"])
)
print(ranked_missions)

section("Optimize in Polars, deliver in pandas")

ranked_missions_pd = ranked_missions.to_pandas()
print(ranked_missions_pd.head(6).to_string(index=False))

round_tripped = pl.from_pandas(ranked_missions_pd)
print("\nRound-tripped schema:")
print(round_tripped.schema)


Rank within each agency
shape: (18, 4)
┌────────────────┬──────────────────┬──────────────────┬─────────────────────────┐
│ agency         ┆ mission_name     ┆ cost_million_usd ┆ cost_rank_within_agency │
│ ---            ┆ ---              ┆ ---              ┆ ---                     │
│ str            ┆ str              ┆ f64              ┆ u32                     │
╞════════════════╪══════════════════╪══════════════════╪═════════════════════════╡
│ CNSA           ┆ Tianwen-1        ┆ 800.0            ┆ 1                       │
│ CNSA           ┆ Chang'e 5        ┆ 200.0            ┆ 2                       │
│ ESA            ┆ JUICE            ┆ 1600.0           ┆ 1                       │
│ ESA            ┆ Solar Orbiter    ┆ 1500.0           ┆ 2                       │
│ ESA/JAXA       ┆ BepiColombo      ┆ 2000.0           ┆ 1                       │
│ …              ┆ …                ┆ …                ┆ …                       │
│ NASA/Roscosmos ┆ ISS Expedition 1 ┆ 150000.0 

In [ ]:
wrap_up = """
Wrap-up prompt
--------------
- Which part of this notebook could pandas also handle comfortably?
- Which part starts to justify Polars because the engine can plan the work?
- If the next tool expects pandas, why is `to_pandas()` a practical escape hatch?

That discussion keeps the focus on workload choice, not library ideology.
"""

print(wrap_up)


Wrap-up prompt
--------------
- Which part of this notebook could pandas also handle comfortably?
- Which part starts to justify Polars because the engine can plan the work?
- If the next tool expects pandas, why is `to_pandas()` a practical escape hatch?

That discussion keeps the focus on workload choice, not library ideology.

